# Image Description Injection

### Setup & imports

In [ ]:
# Setup & imports
from pathlib import Path
import ntpath
from datetime import datetime

### Setup & imports

In [ ]:
# Setup & imports

import os
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "")  

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)


### System Prompt

In [3]:

#prompt builder

def build_prompt(passages, system_prompt):

    user_content = []

    for passage in passages:
        user_content.append({"type": "text",
                             "text": passage})

    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": user_content
        }
    ]
    return chat_prompt


In [4]:
# Modeling & evaluation
def openai_api(lines, system_prompt):
    # Include speech result if speech is enabled
    messages = build_prompt(lines, system_prompt)
    
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=4000,
        temperature=0.7,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False
    )
    
    return completion.choices[0].message.content

In [ ]:
#Image description insertion
gen = os.walk("./LiHua-World")
next(gen)
for x in gen:
    directory = Path(x[0])
    text_files = list(directory.rglob('*.txt'))
    for file in text_files:
        dir = os.path.join("./augmented/conversations", file.parent.absolute().name)
        Path(dir).mkdir(parents=True, exist_ok=True)
        if not os.path.exists(os.path.join(dir, ntpath.split(file.name)[1])):
            with open(file) as file:
                lines = [line.rstrip() for line in file]
                original_datetime_string = lines[0]
                date = datetime.strptime(lines[0][6:], '%Y%m%d_%H:%M')
                lines[0] = date.strftime("%c")

                response = openai_api(lines, "Edit the provided conversation to incorporate an image into the dialogue while maintaining its natural flow and structure. Describe the image clearly when it is introduced and specify who shared it.\n\n---\n\n# Steps\n\n1. **Understand Context**: Analyze the conversation to determine the appropriate moment to include the image based on the subject matter and flow.\n2. **Select Image Context**: Choose an image description relevant to the discussion. Ensure it aligns with the tone and purpose of the conversation.\n3. **Insert Image Transition**: Add a smooth transition that indicates why the image was shared, ensuring it feels natural.\n4. **Describe the Image**: Write a concise but detailed description of the image using sensory and contextual details.\n5. **Specify the Speaker**: Clearly indicate which participant in the conversation shared the image.\n\n---\n\n# Output Format\n\nThe modified conversation should:\n\n- Retain the original dialogue structure and flow.\n- Seamlessly include the image reference and description.\n- Clearly specify the speaker when referencing the shared image.\n- Keep the tone and style consistent with the original conversation.\n\n# Example\n\n**Original Conversation**:  \nPerson A: \"I can't believe how beautiful the sunset was yesterday!\"  \nPerson B: \"I know, it was stunning. The colors were surreal.\"  \nPerson A: \"Absolutely, I've never seen the sky look like that before.\"\n\n**Modified Conversation with an Image**:  \nPerson A: \"I can't believe how beautiful the sunset was yesterday!\"  \nPerson B: \"I know, it was stunning. The colors were surreal.\"  \nPerson A: \"Absolutely, I've never seen the sky look like that before. Here, let me show you a photo I took.\"  \n*Person A: Image: [image of a vibrant sunset, with hues of orange, pink, and purple blending seamlessly across the sky, reflected in a calm ocean below.]*\nPerson B: \"Wow, that's incredible. It almost looks like a painting!\"\n\n(Note: Ensure all modifications to the conversation are consistent with the user’s input and provide enough context for the image inclusion.)")
                with open(os.path.join(dir, ntpath.split(file.name)[1]), "w") as f:
                    split_response = response.splitlines()
                    split_response[0] = original_datetime_string
                    for line in split_response:
                        f.write(line + "\n")